# Entrenamiento y evaluación de modelos para predecir la variable **default** 

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path().resolve().parent.parent / "data/data-09-2025"
data_file = "cleaned_data_default.parquet"

df = pd.read_parquet(DATA_DIR / data_file)
df.head()

,plazo,vinculacion,v_cuota,v_prestamo,s_capital,s_intereses,aportes,garantias,valorgarantia,ctasahorros,...,actividadeconomica,estado_cliente,departamento,sexo,curtotalingresos,curtotalegresos,intestrato,actualizacion,default,puntaje_data
n_credito,,,,,,,,,,,,,,,,,,,,,
003-002-0125852-7,1827,8103,356849.0,15000000.0,12923538.0,123855,7741255,1,7741255,33042953.0,...,asalariados,1,antioquia,0,4597000.0,1500000.0,5.0,1,0,795.0
004-002-0068475-5,1826,1434,2650409.0,100460000.0,31911361.0,263265,4601706,1,4601706,3791115.0,...,asalariados,1,antioquia,0,4597000.0,650000.0,5.0,1,0,836.0
003-002-0122592-9,1826,573,791482.0,30000000.0,23844684.0,261477,530431,1,530431,94435.0,...,asalariados,1,antioquia,0,4400000.0,2000000.0,4.0,0,1,709.0
006-002-0023879-0,2922,1902,2860501.0,176000000.0,113842595.0,1008570,3023534,2,320385440,54841.0,...,educacion_basica_secundaria,1,antioquia,0,22020000.0,1500000.0,4.0,1,0,733.0
006-002-0026159-4,2557,1902,987637.0,50300000.0,38521256.0,317167,1023082,2,320385440,54841.0,...,educacion_basica_secundaria,1,antioquia,0,22020000.0,1500000.0,4.0,1,0,695.0


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12860 entries, 003-002-0125852-7 to 003-002-0119478-4
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   plazo               12860 non-null  int64   
 1   vinculacion         12860 non-null  int64   
 2   v_cuota             12860 non-null  float64 
 3   v_prestamo          12860 non-null  float64 
 4   s_capital           12860 non-null  float64 
 5   s_intereses         12860 non-null  int64   
 6   aportes             12860 non-null  int64   
 7   garantias           12860 non-null  int64   
 8   valorgarantia       12860 non-null  int64   
 9   ctasahorros         12860 non-null  float64 
 10  edad                12860 non-null  float64 
 11  tipoasociado        12860 non-null  int64   
 12  actividadeconomica  12851 non-null  category
 13  estado_cliente      12860 non-null  int64   
 14  departamento        12859 non-null  category
 15  sexo         

In [3]:
df = df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12850 entries, 003-002-0125852-7 to 003-002-0119478-4
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   plazo               12850 non-null  int64   
 1   vinculacion         12850 non-null  int64   
 2   v_cuota             12850 non-null  float64 
 3   v_prestamo          12850 non-null  float64 
 4   s_capital           12850 non-null  float64 
 5   s_intereses         12850 non-null  int64   
 6   aportes             12850 non-null  int64   
 7   garantias           12850 non-null  int64   
 8   valorgarantia       12850 non-null  int64   
 9   ctasahorros         12850 non-null  float64 
 10  edad                12850 non-null  float64 
 11  tipoasociado        12850 non-null  int64   
 12  actividadeconomica  12850 non-null  category
 13  estado_cliente      12850 non-null  int64   
 14  departamento        12850 non-null  category
 15  sexo         

In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["default", "s_capital"]) # s_capital está muy correlacionada con v_prestamo
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=1
)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

Training set size: (10280, 20)
Testing set size: (2570, 20)


In [5]:
from sklearn.metrics import classification_report, confusion_matrix


def print_resultados(model, X_test, y_test):
    print("reporte de clasificación:")
    print(classification_report(y_test, model.predict(X_test)), "\n")
    print("matriz de confusión:")
    print(confusion_matrix(y_test, model.predict(X_test)), "\n")
    feature_importances = pd.Series(model.feature_importances_, index=X_train.columns)
    feature_importances.sort_values(ascending=False, inplace=True)
    print("10 características más importantes:")
    print(feature_importances.head(10))
    return

In [6]:
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score

model = LGBMClassifier(
    objective="binary",
    class_weight="balanced",
    verbose=0,
    random_state=1,
    )

train_x, val_x, train_y, val_y = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=1
)

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

model.fit(
    train_x,
    train_y,
    categorical_feature=cat_vars,
    eval_set=[(val_x, val_y)],
    eval_metric="logloss",
    callbacks=[lgb.early_stopping(stopping_rounds=20)],
)

train_score = f1_score(y_train, model.predict(X_train))
test_score = f1_score(y_test, model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Training until validation scores don't improve for 20 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's binary_logloss: 0.172171
Train score: 0.89
Test score: 0.76


In [7]:
print_resultados(model, X_test, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.93      0.95      2125
           1       0.71      0.83      0.76       445

    accuracy                           0.91      2570
   macro avg       0.84      0.88      0.85      2570
weighted avg       0.92      0.91      0.91      2570
 

matriz de confusión:
[[1972  153]
 [  75  370]] 

10 características más importantes:
s_intereses         432
vinculacion         356
valorgarantia       272
puntaje_data        254
curtotalingresos    241
ctasahorros         235
v_cuota             199
plazo               181
v_prestamo          177
aportes             173
dtype: int32


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
from xgboost import XGBClassifier

te = TargetEncoder()

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("target_encoder", te, cat_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

train_x, val_x, train_y, val_y = train_test_split(
    X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
)

model = XGBClassifier(
    grow_policy="lossguide",
    tree_method="hist",
    early_stopping_rounds=20,
    eval_metric="auc",
    random_state=1,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
)

model.fit(train_x, train_y, eval_set=[(val_x, val_y)], verbose=False)

train_score = f1_score(y_train, model.predict(X_train_processed))
test_score = f1_score(y_test, model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Train score: 0.88
Test score: 0.76


In [9]:
print_resultados(model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.93      0.94      2125
           1       0.70      0.83      0.76       445

    accuracy                           0.91      2570
   macro avg       0.83      0.88      0.85      2570
weighted avg       0.92      0.91      0.91      2570
 

matriz de confusión:
[[1969  156]
 [  75  370]] 

10 características más importantes:
actualizacion       0.543701
estado_cliente      0.083235
tipoasociado        0.056299
garantias           0.032968
edad                0.031550
puntaje_data        0.026722
curtotalingresos    0.025902
v_prestamo          0.024132
aportes             0.020920
s_intereses         0.020424
dtype: float32


In [10]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=1, max_depth=5)
model.fit(X_train_processed, y_train)
train_score = f1_score(y_train, model.predict(X_train_processed))
test_score = f1_score(y_test, model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Train score: 0.69
Test score: 0.64


In [11]:
print_resultados(model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.92      0.95      0.93      2125
           1       0.70      0.59      0.64       445

    accuracy                           0.89      2570
   macro avg       0.81      0.77      0.79      2570
weighted avg       0.88      0.89      0.88      2570
 

matriz de confusión:
[[2013  112]
 [ 182  263]] 

10 características más importantes:
actualizacion       0.280489
tipoasociado        0.250098
edad                0.159845
garantias           0.104361
v_prestamo          0.068376
aportes             0.063838
curtotalingresos    0.046156
s_intereses         0.017301
estado_cliente      0.004859
plazo               0.003476
dtype: float64


In [13]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    auto_class_weights='Balanced',
    verbose=0,
    )

train_x, val_x, train_y, val_y = train_test_split(
    X_train, 
    y_train, 
    test_size=0.2, 
    stratify=y_train, 
    random_state=1
    )

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

model.fit(
    train_x,
    train_y,
    cat_features=cat_vars,
    eval_set=(val_x, val_y),
    early_stopping_rounds=20,
    )

train_score = f1_score(y_train, model.predict(X_train))
test_score = f1_score(y_test, model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Train score: 0.81
Test score: 0.76


In [15]:
print_resultados(model, X_test, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.97      0.91      0.94      2125
           1       0.68      0.87      0.76       445

    accuracy                           0.91      2570
   macro avg       0.82      0.89      0.85      2570
weighted avg       0.92      0.91      0.91      2570
 

matriz de confusión:
[[1941  184]
 [  59  386]] 

10 características más importantes:
actualizacion       12.836181
vinculacion         12.610403
s_intereses         12.052804
tipoasociado        10.212950
ctasahorros          9.324600
puntaje_data         7.275255
valorgarantia        6.127920
curtotalingresos     5.091863
v_cuota              4.634942
aportes              4.533479
dtype: float64


# Sintonización de modelos

### LightGBM

In [17]:
import optuna

def objective(trial):
    param = {
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 50, 500),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),
    }

    categorical_features = X_train.select_dtypes(include=["category"]).columns.tolist()

    train_x, val_x, train_y, val_y = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = LGBMClassifier(
        **param,
        objective="binary",
        force_col_wise=True,
        random_state=1,
        verbose=-1,
    )

    model.fit(
        train_x,
        train_y,
        categorical_feature=categorical_features,
        eval_set=[(val_x, val_y)],
        callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)],
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

lgb_model = LGBMClassifier(
    **study.best_trial.params,
    objective="binary",
    force_col_wise=True,
    random_state=1,
)

categorical_features = X_train.select_dtypes(include=["category"]).columns.tolist()

lgb_model.fit(
    X_train, 
    y_train,
    categorical_feature=categorical_features,
    )

lgb_params = lgb_model.get_params()

train_score = f1_score(y_train, lgb_model.predict(X_train))
test_score = f1_score(y_test, lgb_model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-04 16:44:09,112] A new study created in memory with name: no-name-49b22b20-a13e-4998-aa0f-e1fb2f23d230
[I 2026-03-04 16:44:09,279] Trial 0 finished with value: 0.8121387283236994 and parameters: {'lambda_l1': 0.15640548791888656, 'lambda_l2': 2.5451806420007373e-07, 'num_leaves': 69, 'feature_fraction': 0.5267612265474725, 'bagging_fraction': 0.6333026877113396, 'bagging_freq': 1, 'min_child_samples': 52, 'learning_rate': 0.05561886540427053, 'colsample_bytree': 0.8321675451750088, 'scale_pos_weight': 2.2210113532028704}. Best is trial 0 with value: 0.8121387283236994.
[I 2026-03-04 16:44:09,438] Trial 1 finished with value: 0.8026490066225166 and parameters: {'lambda_l1': 0.10562927010135172, 'lambda_l2': 0.0029120235675680925, 'num_leaves': 454, 'feature_fraction': 0.5254848660511067, 'bagging_fraction': 0.5785131388908792, 'bagging_freq': 3, 'min_child_samples': 47, 'learning_rate': 0.027108404738255095, 'colsample_bytree': 0.6238189223996726, 'scale_pos_weight': 4.261236

Best trial: 0.840 with params: {'lambda_l1': 0.07992269562909127, 'lambda_l2': 0.04801021576226848, 'num_leaves': 317, 'feature_fraction': 0.5643072189125093, 'bagging_fraction': 0.9737951944951198, 'bagging_freq': 6, 'min_child_samples': 18, 'learning_rate': 0.10710129890663363, 'colsample_bytree': 0.9765582153234154, 'scale_pos_weight': 6.774605567601114}
Train score: 1.00
Test score: 0.80


In [18]:
print_resultados(lgb_model, X_test, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.95      0.97      0.96      2125
           1       0.83      0.77      0.80       445

    accuracy                           0.93      2570
   macro avg       0.89      0.87      0.88      2570
weighted avg       0.93      0.93      0.93      2570
 

matriz de confusión:
[[2055   70]
 [ 102  343]] 

10 características más importantes:
v_cuota             2824
vinculacion         2758
curtotalingresos    2608
valorgarantia       2546
ctasahorros         2209
s_intereses         2077
aportes             2034
edad                1896
v_prestamo          1841
puntaje_data        1823
dtype: int32


### XGBoost

In [19]:
def objective(trial):
    te = TargetEncoder()

    cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("target_encoder", te, cat_vars),
        ],
        remainder="passthrough",
    )

    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    X_test_processed = preprocessor.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 1e-2, 5e-1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 1e2, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 1e2, log=True),
        "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
        "gamma": trial.suggest_float("gamma", 0, 2),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 10),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),
    }

    model = XGBClassifier(
        grow_policy="lossguide",
        tree_method="hist",
        early_stopping_rounds=20,
        eval_metric="auc",
        random_state=1,
    )

    model.fit(train_x, train_y, eval_set=[(val_x, val_y)], verbose=False)

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

xgb_model = XGBClassifier(
    **study.best_trial.params,
    grow_policy="lossguide",
    tree_method="hist",
    # early_stopping_rounds=20,
    # eval_metric="auc",
    random_state=1,
)

te = TargetEncoder()

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("target_encoder", te, cat_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

xgb_model.fit(X_train_processed, y_train)
xgb_params = xgb_model.get_params()

train_score = f1_score(y_train, xgb_model.predict(X_train_processed))
test_score = f1_score(y_test, xgb_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-04 16:45:49,150] A new study created in memory with name: no-name-e04ed56f-5315-4dff-9c97-d453904b193d
[I 2026-03-04 16:45:49,387] Trial 0 finished with value: 0.7933130699088146 and parameters: {'max_depth': 12, 'learning_rate': 0.05844147591130633, 'n_estimators': 450, 'subsample': 0.8635449086738746, 'colsample_bytree': 0.8049230922205699, 'reg_alpha': 0.00470142782625201, 'reg_lambda': 1.0520590491470543, 'max_delta_step': 0, 'gamma': 0.08039662962116578, 'min_child_weight': 8, 'scale_pos_weight': 5.436460261839623}. Best is trial 0 with value: 0.7933130699088146.
[I 2026-03-04 16:45:49,522] Trial 1 finished with value: 0.7981510015408321 and parameters: {'max_depth': 11, 'learning_rate': 0.32661602210284146, 'n_estimators': 204, 'subsample': 0.5464459167460586, 'colsample_bytree': 0.8704911075036188, 'reg_alpha': 0.035990283425255114, 'reg_lambda': 0.2619398233361071, 'max_delta_step': 9, 'gamma': 1.0984473148557055, 'min_child_weight': 5, 'scale_pos_weight': 9.58150550

Best trial: 0.817 with params: {'max_depth': 6, 'learning_rate': 0.06255103857190089, 'n_estimators': 136, 'subsample': 0.777634354919278, 'colsample_bytree': 0.7781291927839915, 'reg_alpha': 0.4681261888647303, 'reg_lambda': 1.3873583258645417, 'max_delta_step': 0, 'gamma': 0.7405008464964599, 'min_child_weight': 6, 'scale_pos_weight': 5.0840672055973135}
Train score: 0.86
Test score: 0.77


In [20]:
print_resultados(xgb_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.97      0.92      0.95      2125
           1       0.70      0.86      0.77       445

    accuracy                           0.91      2570
   macro avg       0.84      0.89      0.86      2570
weighted avg       0.92      0.91      0.92      2570
 

matriz de confusión:
[[1963  162]
 [  64  381]] 

10 características más importantes:
actualizacion       0.520422
estado_cliente      0.074626
tipoasociado        0.064578
curtotalingresos    0.038703
garantias           0.037538
edad                0.032584
puntaje_data        0.029911
v_prestamo          0.026780
valorgarantia       0.024765
s_intereses         0.023605
dtype: float32


### Random Forest

In [22]:
from sklearn.ensemble import RandomForestClassifier

def objective(trial):
    te = TargetEncoder()

    cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("target_encoder", te, cat_vars),
        ],
        remainder="passthrough",
    )

    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    X_test_processed = preprocessor.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = RandomForestClassifier(class_weight="balanced", random_state=1)

    param = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_categorical(
            "max_features", ["sqrt", "log2", None]
        ),
    }

    model.fit(
        train_x,
        train_y,
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

rf_model = RandomForestClassifier(
    **study.best_trial.params,
    class_weight="balanced",
    random_state=1,
)

te = TargetEncoder()

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("target_encoder", te, cat_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

rf_model.fit(X_train_processed, y_train)
rf_params = rf_model.get_params()

train_score = f1_score(y_train, rf_model.predict(X_train_processed))
test_score = f1_score(y_test, rf_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-04 16:47:44,227] A new study created in memory with name: no-name-d4aa0ca5-9af8-4cdb-86b7-d0c8c5e16ff4
[I 2026-03-04 16:47:45,394] Trial 0 finished with value: 0.743801652892562 and parameters: {'n_estimators': 207, 'max_depth': 4, 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.743801652892562.
[I 2026-03-04 16:47:46,600] Trial 1 finished with value: 0.7458745874587459 and parameters: {'n_estimators': 336, 'max_depth': 3, 'min_samples_split': 15, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 1 with value: 0.7458745874587459.
[I 2026-03-04 16:47:47,822] Trial 2 finished with value: 0.7475083056478405 and parameters: {'n_estimators': 480, 'max_depth': 7, 'min_samples_split': 12, 'min_samples_leaf': 19, 'max_features': 'sqrt'}. Best is trial 2 with value: 0.7475083056478405.
[I 2026-03-04 16:47:49,044] Trial 3 finished with value: 0.7574750830564784 and parameters: {'n_estimators': 108, 'max_depth': 10, 'min

Best trial: 0.764 with params: {'n_estimators': 129, 'max_depth': 12, 'min_samples_split': 20, 'min_samples_leaf': 19, 'max_features': 'log2'}
Train score: 0.80
Test score: 0.75


In [23]:
print_resultados(rf_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.97      0.91      0.94      2125
           1       0.67      0.84      0.75       445

    accuracy                           0.90      2570
   macro avg       0.82      0.88      0.84      2570
weighted avg       0.91      0.90      0.91      2570
 

matriz de confusión:
[[1942  183]
 [  70  375]] 

10 características más importantes:
actualizacion       0.220362
tipoasociado        0.148926
garantias           0.142416
curtotalingresos    0.075839
edad                0.070394
puntaje_data        0.066515
v_prestamo          0.063216
estado_cliente      0.060703
valorgarantia       0.040993
s_intereses         0.039706
dtype: float64


### CatBoost

In [ ]:
def objective(trial):
    param = {
        'iterations': trial.suggest_int('iterations', 50, 1000),
        'depth': trial.suggest_int('depth', 3, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 1, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 1.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_strength': trial.suggest_float('random_strength', 1e-3, 10, log=True),
        }

    categorical_features = X_train.select_dtypes(include=["category"]).columns.tolist()

    train_x, val_x, train_y, val_y = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = CatBoostClassifier(
        auto_class_weights='Balanced',
        verbose=0,
        )


    model.fit(
        train_x,
        train_y,
        cat_features=categorical_features,
        eval_set=(val_x, val_y),
        early_stopping_rounds=20,
        )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

categorical_features = X_train.select_dtypes(include=["category"]).columns.tolist()

cat_model = CatBoostClassifier(
    **study.best_trial.params,
    auto_class_weights='Balanced',
    verbose=0,
    )

cat_model.fit(
    X_train, 
    y_train,
    cat_features=categorical_features,
    )

cat_params = cat_model.get_params()

train_score = f1_score(y_train, cat_model.predict(X_train))
test_score = f1_score(y_test, cat_model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")


[I 2026-03-04 16:55:47,756] A new study created in memory with name: no-name-57fc5c4b-6fef-4180-81f4-5bb41bd853d3
[I 2026-03-04 16:55:58,362] Trial 0 finished with value: 0.7875 and parameters: {'iterations': 881, 'depth': 3, 'learning_rate': 0.032853740082395874, 'l2_leaf_reg': 0.03304553310041836, 'min_child_samples': 37, 'subsample': 0.810472124083289, 'colsample_bylevel': 0.7368834199954902, 'border_count': 145, 'random_strength': 4.50857078016408}. Best is trial 0 with value: 0.7875.
[I 2026-03-04 16:56:10,189] Trial 1 finished with value: 0.7875 and parameters: {'iterations': 557, 'depth': 6, 'learning_rate': 0.1340266537538306, 'l2_leaf_reg': 0.019104764831143796, 'min_child_samples': 21, 'subsample': 0.9738399875900019, 'colsample_bylevel': 0.6750950290788504, 'border_count': 49, 'random_strength': 0.0038967257171566017}. Best is trial 0 with value: 0.7875.
[I 2026-03-04 16:56:21,909] Trial 2 finished with value: 0.7875 and parameters: {'iterations': 227, 'depth': 5, 'learning_

Best trial: 0.787 with params: {'iterations': 881, 'depth': 3, 'learning_rate': 0.032853740082395874, 'l2_leaf_reg': 0.03304553310041836, 'min_child_samples': 37, 'subsample': 0.810472124083289, 'colsample_bylevel': 0.7368834199954902, 'border_count': 145, 'random_strength': 4.50857078016408}
Train score: 0.78
Test score: 0.76


In [30]:
print_resultados(cat_model, X_test, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.97      0.91      0.94      2125
           1       0.66      0.88      0.76       445

    accuracy                           0.90      2570
   macro avg       0.82      0.89      0.85      2570
weighted avg       0.92      0.90      0.91      2570
 

matriz de confusión:
[[1925  200]
 [  52  393]] 

10 características más importantes:
vinculacion         20.616924
s_intereses         12.851701
actualizacion       12.332077
ctasahorros         10.163586
valorgarantia        6.437755
tipoasociado         6.244777
puntaje_data         5.121384
curtotalingresos     4.946668
plazo                4.027214
v_cuota              3.820395
dtype: float64


# Modelos con Skrub

### LightGBM

In [48]:
def print_resultados_skrub(model, X_test, y_test):
    print("reporte de clasificación:")
    print(classification_report(y_test, model.predict(X_test)), "\n")
    print("matriz de confusión:")
    print(confusion_matrix(y_test, model.predict(X_test)), "\n")
    feature_importances = pd.Series(
        model.feature_importances_, index=X_train_processed.columns.to_list()
    )
    feature_importances.sort_values(ascending=False, inplace=True)
    print("10 características más importantes:")
    print(feature_importances.head(10))
    return

In [39]:
import warnings

from skrub import TableVectorizer

warnings.filterwarnings("ignore")


def objective(trial):
    param = {
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 50, 500),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),
    }

    vectorizer = TableVectorizer()

    X_train_processed = vectorizer.fit_transform(X_train, y_train)
    X_test_processed = vectorizer.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = LGBMClassifier(
        **param,
        objective="binary",
        force_col_wise=True,
        random_state=1,
        verbose=-1,
    )

    model.fit(
        train_x,
        train_y,
        eval_set=[(val_x, val_y)],
        callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)],
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

vectorizer = TableVectorizer()

X_train_processed = vectorizer.fit_transform(X_train, y_train)
X_test_processed = vectorizer.transform(X_test)

sk_lgb_model = LGBMClassifier(
    **study.best_trial.params,
    objective="binary",
    force_col_wise=True,
    random_state=1,
)

sk_lgb_model.fit(X_train_processed, y_train)
sk_lgb_params = sk_lgb_model.get_params()

train_score = f1_score(y_train, sk_lgb_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_lgb_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-04 17:11:43,921] A new study created in memory with name: no-name-6d96e6dc-5361-4e9b-80ca-a8f6a31776c7
[I 2026-03-04 17:11:44,597] Trial 0 finished with value: 0.8006134969325154 and parameters: {'lambda_l1': 0.04881693419636028, 'lambda_l2': 0.0017347191487348565, 'num_leaves': 125, 'feature_fraction': 0.7383761969724081, 'bagging_fraction': 0.7555444164026384, 'bagging_freq': 1, 'min_child_samples': 6, 'learning_rate': 0.018476155693641923, 'colsample_bytree': 0.7622667654127175, 'scale_pos_weight': 2.3161720545185744}. Best is trial 0 with value: 0.8006134969325154.
[I 2026-03-04 17:11:45,254] Trial 1 finished with value: 0.7978290366350068 and parameters: {'lambda_l1': 5.6761050027326934e-08, 'lambda_l2': 0.0013397822276749948, 'num_leaves': 477, 'feature_fraction': 0.6007026105911024, 'bagging_fraction': 0.7882615840097167, 'bagging_freq': 7, 'min_child_samples': 40, 'learning_rate': 0.011220518498576484, 'colsample_bytree': 0.5026932505110816, 'scale_pos_weight': 9.211

Best trial: 0.838 with params: {'lambda_l1': 0.22177994724968644, 'lambda_l2': 0.0010804905074159258, 'num_leaves': 89, 'feature_fraction': 0.7817752668662897, 'bagging_fraction': 0.933201759910495, 'bagging_freq': 1, 'min_child_samples': 35, 'learning_rate': 0.09133216360913808, 'colsample_bytree': 0.9399692233371545, 'scale_pos_weight': 3.647620367169822}
Train score: 0.99
Test score: 0.78


In [49]:
print_resultados_skrub(sk_lgb_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.95      0.96      0.95      2125
           1       0.79      0.77      0.78       445

    accuracy                           0.92      2570
   macro avg       0.87      0.86      0.87      2570
weighted avg       0.92      0.92      0.92      2570
 

matriz de confusión:
[[2033   92]
 [ 102  343]] 

10 características más importantes:
vinculacion         986
s_intereses         978
curtotalingresos    801
ctasahorros         763
valorgarantia       741
v_cuota             683
puntaje_data        673
v_prestamo          549
edad                542
aportes             538
dtype: int32


### Skrub con modelo por defecto

In [35]:
from skrub import tabular_pipeline


def objective(trial):
    param = {
        "histgradientboostingclassifier__learning_rate": trial.suggest_float(
            "histgradientboostingclassifier__learning_rate", 0.01, 0.2
        ),  # noqa: E501
        "histgradientboostingclassifier__max_iter": trial.suggest_int(
            "histgradientboostingclassifier__max_iter", 100, 500
        ),  # noqa: E501
        "histgradientboostingclassifier__max_leaf_nodes": trial.suggest_int(
            "histgradientboostingclassifier__max_leaf_nodes", 20, 100
        ),  # noqa: E501
        "histgradientboostingclassifier__min_samples_leaf": trial.suggest_int(
            "histgradientboostingclassifier__min_samples_leaf", 1, 10
        ),  # noqa: E501
        "histgradientboostingclassifier__l2_regularization": trial.suggest_float(
            "histgradientboostingclassifier__l2_regularization", 1e-3, 10.0, log=True
        ),  # noqa: E501
        "histgradientboostingclassifier__early_stopping": trial.suggest_categorical(
            "histgradientboostingclassifier__early_stopping", [True, False]
        ),  # noqa: E501
    }

    train_x, val_x, train_y, val_y = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = tabular_pipeline(estimator="classifier")

    model.fit(
        train_x,
        train_y,
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_params = study.best_trial.params
sk_params = {
    k.replace("histgradientboostingclassifier__", ""): v for k, v in sk_params.items()
}  # noqa: E501
sk_model = tabular_pipeline(estimator="classifier")
sk_model["histgradientboostingclassifier"].set_params(**sk_params)
sk_model.fit(X_train, y_train)
sk_params = sk_model.get_params()

train_score = f1_score(y_train, sk_model.predict(X_train))
test_score = f1_score(y_test, sk_model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-04 17:03:08,090] A new study created in memory with name: no-name-ba72d564-fe71-4076-a1d5-9d6e28129394
[I 2026-03-04 17:03:08,726] Trial 0 finished with value: 0.8072837632776935 and parameters: {'histgradientboostingclassifier__learning_rate': 0.1890653691955483, 'histgradientboostingclassifier__max_iter': 292, 'histgradientboostingclassifier__max_leaf_nodes': 71, 'histgradientboostingclassifier__min_samples_leaf': 8, 'histgradientboostingclassifier__l2_regularization': 0.8262466586050926, 'histgradientboostingclassifier__early_stopping': False}. Best is trial 0 with value: 0.8072837632776935.
[I 2026-03-04 17:03:09,361] Trial 1 finished with value: 0.8072837632776935 and parameters: {'histgradientboostingclassifier__learning_rate': 0.16772012062886052, 'histgradientboostingclassifier__max_iter': 153, 'histgradientboostingclassifier__max_leaf_nodes': 61, 'histgradientboostingclassifier__min_samples_leaf': 10, 'histgradientboostingclassifier__l2_regularization': 1.1762667269

Best trial: 0.817 with params: {'histgradientboostingclassifier__learning_rate': 0.07753587262623715, 'histgradientboostingclassifier__max_iter': 325, 'histgradientboostingclassifier__max_leaf_nodes': 68, 'histgradientboostingclassifier__min_samples_leaf': 3, 'histgradientboostingclassifier__l2_regularization': 1.5065916219875337, 'histgradientboostingclassifier__early_stopping': False}
Train score: 1.00
Test score: 0.78


In [36]:
print("reporte de clasificación:")
print(classification_report(y_test, sk_model.predict(X_test)), "\n")
print("matriz de confusión:")
print(confusion_matrix(y_test, sk_model.predict(X_test)), "\n")

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.94      0.97      0.96      2125
           1       0.84      0.72      0.78       445

    accuracy                           0.93      2570
   macro avg       0.89      0.84      0.87      2570
weighted avg       0.93      0.93      0.93      2570
 

matriz de confusión:
[[2066   59]
 [ 126  319]] 



### XGBoost

In [37]:
def objective(trial):
    vectorizer = TableVectorizer()

    X_train_processed = vectorizer.fit_transform(X_train, y_train)
    X_test_processed = vectorizer.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 1e-2, 5e-1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 1e2, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 1e2, log=True),
        "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
        "gamma": trial.suggest_float("gamma", 0, 2),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 10),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),
    }

    model = XGBClassifier(
        grow_policy="lossguide",
        tree_method="hist",
        early_stopping_rounds=20,
        eval_metric="auc",
        random_state=1,
    )

    model.fit(train_x, train_y, eval_set=[(val_x, val_y)], verbose=False)

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_xgb_model = XGBClassifier(
    **study.best_trial.params,
    grow_policy="lossguide",
    tree_method="hist",
    random_state=1,
)

vectorizer = TableVectorizer()

X_train_processed = vectorizer.fit_transform(X_train, y_train)
X_test_processed = vectorizer.transform(X_test)

sk_xgb_model.fit(X_train_processed, y_train)
sk_xgb_params = xgb_model.get_params()

train_score = f1_score(y_train, sk_xgb_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_xgb_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-04 17:05:19,321] A new study created in memory with name: no-name-0750bea1-363a-4782-aab9-97998f19f3fc
[I 2026-03-04 17:05:19,987] Trial 0 finished with value: 0.796969696969697 and parameters: {'max_depth': 12, 'learning_rate': 0.23833116902300172, 'n_estimators': 223, 'subsample': 0.932855385574429, 'colsample_bytree': 0.5140894387474904, 'reg_alpha': 0.0019271367519803338, 'reg_lambda': 0.2277105448400874, 'max_delta_step': 10, 'gamma': 1.4125061709945044, 'min_child_weight': 7, 'scale_pos_weight': 2.5163389925026483}. Best is trial 0 with value: 0.796969696969697.
[I 2026-03-04 17:05:20,552] Trial 1 finished with value: 0.8023774145616642 and parameters: {'max_depth': 14, 'learning_rate': 0.010783138312219118, 'n_estimators': 421, 'subsample': 0.9997360266816926, 'colsample_bytree': 0.5657230047845435, 'reg_alpha': 64.85856969295655, 'reg_lambda': 51.554242341823226, 'max_delta_step': 5, 'gamma': 0.539872401429883, 'min_child_weight': 8, 'scale_pos_weight': 1.31318044907

Best trial: 0.812 with params: {'max_depth': 6, 'learning_rate': 0.044504971836058206, 'n_estimators': 102, 'subsample': 0.5000961874787534, 'colsample_bytree': 0.8911323807233601, 'reg_alpha': 0.1498363033230646, 'reg_lambda': 0.036306445299066716, 'max_delta_step': 7, 'gamma': 0.7784297102146946, 'min_child_weight': 9, 'scale_pos_weight': 5.043658629240945}
Train score: 0.80
Test score: 0.75


In [50]:
print_resultados_skrub(sk_xgb_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.97      0.91      0.94      2125
           1       0.67      0.88      0.76       445

    accuracy                           0.90      2570
   macro avg       0.82      0.89      0.85      2570
weighted avg       0.92      0.90      0.91      2570
 

matriz de confusión:
[[1930  195]
 [  55  390]] 

10 características más importantes:
actualizacion            0.293831
ctasahorros              0.047315
actividadeconomica_10    0.035087
tipoasociado             0.033159
s_intereses              0.026943
valorgarantia            0.024558
curtotalingresos         0.022407
actividadeconomica_29    0.021982
actividadeconomica_08    0.020833
puntaje_data             0.020210
dtype: float32


### Random Forest

In [ ]:
def objective(trial):
    vectorizer = TableVectorizer()

    X_train_processed = vectorizer.fit_transform(X_train, y_train)
    X_test_processed = vectorizer.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = RandomForestClassifier(class_weight="balanced", random_state=1)

    param = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_categorical(
            "max_features", ["sqrt", "log2", None]
        ),
    }

    model.fit(
        train_x,
        train_y,
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_rf_model = RandomForestClassifier(
    **study.best_trial.params,
    class_weight="balanced",
    random_state=1,
)

vectorizer = TableVectorizer()

X_train_processed = vectorizer.fit_transform(X_train, y_train)
X_test_processed = vectorizer.transform(X_test)

sk_rf_model.fit(X_train_processed, y_train)
sk_rf_params = sk_rf_model.get_params()

train_score = f1_score(y_train, sk_rf_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_rf_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-02-27 13:51:27,245] A new study created in memory with name: no-name-c87bf1f0-9e9a-4ded-8d51-99f3a1c5467d
[I 2026-02-27 13:51:30,806] Trial 0 finished with value: 0.6915254237288135 and parameters: {'n_estimators': 402, 'max_depth': 15, 'min_samples_split': 4, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.6915254237288135.
[I 2026-02-27 13:51:34,792] Trial 1 finished with value: 0.6825938566552902 and parameters: {'n_estimators': 295, 'max_depth': 5, 'min_samples_split': 8, 'min_samples_leaf': 7, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.6915254237288135.
[I 2026-02-27 13:51:38,086] Trial 2 finished with value: 0.6802721088435374 and parameters: {'n_estimators': 400, 'max_depth': 8, 'min_samples_split': 10, 'min_samples_leaf': 12, 'max_features': 'log2'}. Best is trial 0 with value: 0.6915254237288135.
[I 2026-02-27 13:51:41,802] Trial 3 finished with value: 0.6870748299319728 and parameters: {'n_estimators': 403, 'max_depth': 15, 'm

Best trial: 0.702 with params: {'n_estimators': 210, 'max_depth': 9, 'min_samples_split': 12, 'min_samples_leaf': 4, 'max_features': 'sqrt'}
Train score: 0.80
Test score: 0.72


In [30]:
print_resultados_skrub(sk_rf_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.95      0.92      0.94      2126
           1       0.67      0.79      0.72       446

    accuracy                           0.90      2572
   macro avg       0.81      0.85      0.83      2572
weighted avg       0.90      0.90      0.90      2572
 

matriz de confusión:
[[1948  178]
 [  92  354]] 

10 características más importantes:
actualizacion       0.186592
ctasahorros         0.137324
s_intereses         0.129949
curtotalingresos    0.088720
puntaje_data        0.069153
valorgarantia       0.064892
tipoasociado        0.061050
vinculacion         0.060561
aportes             0.048338
v_cuota             0.032859
dtype: float64


### CatBoost

In [ ]:
def objective(trial):

    vectorizer = TableVectorizer()

    X_train_processed = vectorizer.fit_transform(X_train, y_train)
    X_test_processed = vectorizer.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    param = {
        'iterations': trial.suggest_int('iterations', 50, 1000),
        'depth': trial.suggest_int('depth', 3, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 1, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 1.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_strength': trial.suggest_float('random_strength', 1e-3, 10, log=True),
        }

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = CatBoostClassifier(
        auto_class_weights='Balanced',
        verbose=0,
        )


    model.fit(
        train_x,
        train_y,
        eval_set=(val_x, val_y),
        early_stopping_rounds=20,
        )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

vectorizer = TableVectorizer()

X_train_processed = vectorizer.fit_transform(X_train, y_train)
X_test_processed = vectorizer.transform(X_test)

sk_cat_model = CatBoostClassifier(
    **study.best_trial.params,
    auto_class_weights='Balanced',
    verbose=0,
    )

sk_cat_model.fit(
    X_train_processed, 
    y_train,
    )

sk_cat_params = cat_model.get_params()

train_score = f1_score(y_train, sk_cat_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_cat_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")


[I 2026-03-04 17:16:44,009] A new study created in memory with name: no-name-8b79f31e-3cdd-4fa4-b502-9a31b15e0fff
[I 2026-03-04 17:16:45,787] Trial 0 finished with value: 0.7918781725888325 and parameters: {'iterations': 433, 'depth': 5, 'learning_rate': 0.06990571729118422, 'l2_leaf_reg': 0.0037515770729069837, 'min_child_samples': 15, 'subsample': 0.96152561799823, 'colsample_bylevel': 0.960336063234073, 'border_count': 223, 'random_strength': 0.5477789646431492}. Best is trial 0 with value: 0.7918781725888325.
[I 2026-03-04 17:16:47,658] Trial 1 finished with value: 0.8010335917312662 and parameters: {'iterations': 391, 'depth': 5, 'learning_rate': 0.046458840311835295, 'l2_leaf_reg': 0.01855525550263447, 'min_child_samples': 4, 'subsample': 0.9571789335914904, 'colsample_bylevel': 0.9701491952261021, 'border_count': 114, 'random_strength': 0.010498834911763575}. Best is trial 1 with value: 0.8010335917312662.


Best trial: 0.801 with params: {'iterations': 391, 'depth': 5, 'learning_rate': 0.046458840311835295, 'l2_leaf_reg': 0.01855525550263447, 'min_child_samples': 4, 'subsample': 0.9571789335914904, 'colsample_bylevel': 0.9701491952261021, 'border_count': 114, 'random_strength': 0.010498834911763575}
Train score: 0.86
Test score: 0.76


In [51]:
print_resultados_skrub(sk_cat_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.97      0.92      0.94      2125
           1       0.69      0.84      0.76       445

    accuracy                           0.91      2570
   macro avg       0.83      0.88      0.85      2570
weighted avg       0.92      0.91      0.91      2570
 

matriz de confusión:
[[1959  166]
 [  69  376]] 

10 características más importantes:
vinculacion         13.362444
s_intereses         12.600714
ctasahorros          8.542832
valorgarantia        8.444451
actualizacion        8.265318
v_cuota              6.948601
curtotalingresos     6.285258
puntaje_data         6.022490
tipoasociado         6.006858
plazo                5.687850
dtype: float64


# Guardado de modelos y parámetos

In [31]:
# import json

# import joblib

# MODELS_DIR = Path().resolve().parent / "models"
# models = {
#     "lgb": (lgb_model, lgb_params),
#     "xgb": (xgb_model, xgb_params),
#     "rf": (rf_model, rf_params),
#     "lr": (lr_model, lr_params),
#     "sk_lgb": (sk_lgb_model, sk_lgb_params),
#     "sk_xgb": (sk_xgb_model, sk_xgb_params),
#     "sk_rf": (sk_rf_model, sk_rf_params),
#     "sk_lr": (sk_lr_model, sk_lr_params),
#     }

# # Guardar modelos en formato joblib y parámetros en formato JSON    

# for name, (model, params) in models.items():
#     joblib.dump(model, MODELS_DIR / f"{name}_model.joblib")
#     with open(MODELS_DIR / f"{name}_params.json", "w") as f:
#         json.dump(params, f)